In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)



In [ ]:
df = pd.read_csv('fifa_player_performance_market_value.csv')


print(f'Shape: {df.shape}')

In [ ]:
# Part 1:  Dataset Understanding and Exploration
# Displaying the first 12 rows to get an initial sense of the data structure,
# column names, value formats, and any immediately visible quality issues.
print('=== First 12 Rows ===')
df.head(12)

In [ ]:
# Displaying the last 12 rows to check for any trailing anomalies
# or inconsistencies at the end of the dataset.
print('=== Last 12 Rows ===')
df.tail(12)

In [ ]:
# Knowing the dataset dimensions is the first step in scoping the analysis.
rows, cols = df.shape
print(f'Total Rows    : {rows}')
print(f'Total Columns : {cols}')

In [ ]:
# Identifying data types helps us determine which columns need encoding
# (categorical) and which are ready for numerical analysis.
print('=== Column Names and Data Types ===')
dtype_df = pd.DataFrame({
    'Column'   : df.columns,
    'Data Type': df.dtypes.values
})
print(dtype_df.to_string(index=False))

In [ ]:
# df.info() provides a compact overview: column names, non-null counts,
# and data types — essential for spotting missing data at a glance.
print('=== Dataset Summary ===')
df.info()

In [ ]:
# The target variable is 'market_value_million_eur'.
# We bin it into 3 equal-frequency categories (Low / Medium / High)


df['value_category'] = pd.qcut(
    df['market_value_million_eur'],
    q=3,
    labels=['Low Value', 'Medium Value', 'High Value'],
    duplicates='drop'
)

print('Target Variable : market_value_million_eur')
print('Binned Target   : value_category (Low / Medium / High Value)')
print()

dist     = df['value_category'].value_counts().sort_index()
dist_pct = df['value_category'].value_counts(normalize=True).mul(100).round(2).sort_index()

print('=== Class Distribution ===')
print(pd.DataFrame({'Count': dist, 'Percentage (%)': dist_pct}))

# Visualization
plt.figure(figsize=(7, 4))
bars = plt.bar(dist.index, dist.values, color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='black')
for bar, pct in zip(bars, dist_pct.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f'{pct}%', ha='center', va='bottom', fontsize=10)
plt.title('Target Variable Class Distribution\n(Player Market Value Category)', fontsize=13)
plt.xlabel('Value Category')
plt.ylabel('Number of Players')
plt.tight_layout()
plt.show()

print()
print('Insight: The three classes are nearly perfectly balanced (~33% each),')
print('meaning class imbalance is not a major concern for this target variable.')

In [ ]:
# We selected 'position' as our categorical feature.
# It has 9 distinct values (ST, RW, LW, CM, CDM, CB, LB, RB, GK)
# and is directly relevant to player value and performance patterns.

cat_feature = 'position'

print(f'=== Distinct Values in "{cat_feature}" ===')
distinct_vals = sorted(df[cat_feature].dropna().unique())
print(f'Total distinct values: {len(distinct_vals)}')
print(distinct_vals)

print()
most_frequent = df[cat_feature].mode()[0]
freq_count    = df[cat_feature].value_counts()[most_frequent]
print(f'Most Frequent Value : "{most_frequent}" — appears {freq_count} times '
      f'({freq_count / len(df) * 100:.1f}% of all players)')

print()
print('=== Full Value Counts ===')
vc = df[cat_feature].value_counts()
print(vc)

# Visualization
plt.figure(figsize=(8, 4))
vc.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Player Count by Position', fontsize=13)
plt.xlabel('Position')
plt.ylabel('Number of Players')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# We exclude 'player_id' as it is just an identifier with no analytical value.
# Comparing mean vs. median helps detect skewness in each feature.

num_cols = ['age', 'overall_rating', 'potential_rating', 'matches_played',
            'goals', 'assists', 'minutes_played', 'market_value_million_eur',
            'contract_years_left']

print('=== Mean, Median, Standard Deviation ===')
stats_df = pd.DataFrame({
    'Mean'   : df[num_cols].mean(),
    'Median' : df[num_cols].median(),
    'Std Dev': df[num_cols].std()
})
print(stats_df)

print()
print('=== Percentiles (20%, 50%, 75%) ===')
percentiles = df[num_cols].quantile([0.20, 0.50, 0.75])
percentiles.index = ['20th Percentile', '50th Percentile', '75th Percentile']
print(percentiles)

In [ ]:
# Detecting missing values is critical before any modelling step.
# Even a seemingly clean dataset should be explicitly verified.

missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing (%)': missing_pct})
cols_with_missing = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print(f'Total columns checked : {len(df.columns)}')
print(f'Columns with missing  : {len(cols_with_missing)}')
print()

if cols_with_missing.empty:
    print('No missing values found in any column. The dataset is complete.')
else:
    print('=== Columns with Missing Values ===')
    print(cols_with_missing)
    plt.figure(figsize=(10, 5))
    cols_with_missing['Missing Count'].plot(kind='bar', color='salmon', edgecolor='black')
    plt.title('Missing Values per Column')
    plt.xlabel('Column')
    plt.ylabel('Missing Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Duplicate rows can distort model training by overrepresenting certain players.
# We detect and remove them to ensure data integrity.

num_duplicates = df.duplicated().sum()
print(f'Number of Duplicate Records Found: {num_duplicates}')

if num_duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Duplicates removed. New dataset shape: {df.shape}')
else:
    print('No duplicates found. Dataset integrity confirmed.')
    print(f'Final shape: {df.shape}')

In [ ]:
#Part 2: Data Preparation
# We filter to keep only players with at least 10 matches played.
# Players with fewer appearances may be injured, youth squad, or inactive —
# their stats (goals, assists, minutes) would be unreliable and skew the analysis.

print(f'Shape before filtering: {df.shape}')
df = df[df['matches_played'] >= 10].reset_index(drop=True)
print(f'Shape after filtering : {df.shape}')
print(f'Players removed       : {2800 - len(df)}')

In [ ]:
# Encoding strategy:
# - 'injury_prone' → Binary encoding (No=0, Yes=1) since it has only 2 values
# - 'transfer_risk_level' → Ordinal encoding (Low=0, Medium=1, High=2) since it has a natural order
# - 'position', 'nationality', 'club' → One-Hot Encoding since they are nominal with no order

from sklearn.preprocessing import LabelEncoder

# Binary encoding for injury_prone
df['injury_prone_encoded'] = df['injury_prone'].map({'No': 0, 'Yes': 1})
print('injury_prone encoded (Binary):')
print(df[['injury_prone', 'injury_prone_encoded']].value_counts().reset_index())

print()
# Ordinal encoding for transfer_risk_level
df['transfer_risk_encoded'] = df['transfer_risk_level'].map({'Low': 0, 'Medium': 1, 'High': 2})
print('transfer_risk_level encoded (Ordinal):')
print(df[['transfer_risk_level', 'transfer_risk_encoded']].value_counts().reset_index())
 
print()
# One-Hot Encoding for position, nationality, club
df = pd.get_dummies(df, columns=['position', 'nationality', 'club'], drop_first=False)
print(f'Shape after One-Hot Encoding: {df.shape}')
print('New columns added:', [c for c in df.columns if any(c.startswith(p) for p in ['position_', 'nationality_', 'club_'])])

In [ ]:

from sklearn.preprocessing import StandardScaler

# StandardScaler standardizes features to mean=0 and std=1.
# This is essential for distance-based algorithms (KNN, SVM) and
# prevents features with large ranges (like minutes_played) from
# dominating those with small ranges (like contract_years_left).

num_cols = ['age', 'overall_rating', 'potential_rating', 'matches_played',
            'goals', 'assists', 'minutes_played', 'market_value_million_eur',
            'contract_years_left']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[num_cols] = scaler.fit_transform(df[num_cols])

print('=== Before Scaling (sample means) ===')
print(df[num_cols].mean().round(2))

print()
print('=== After Scaling (should be ~0 mean, ~1 std) ===')
print(df_scaled[num_cols].mean().round(4))
print()
print(df_scaled[num_cols].std().round(4))

In [ ]:
# We bin 'age' into 5 equal-width bins.
# Age is meaningful to divide this way — it creates natural football career
# stages: Youth, Early Career, Prime, Experienced, Veteran.
# Equal-width (cut) is appropriate here since age has a clear bounded range (17–39).

df['age_bin'] = pd.cut(
    df['age'],
    bins=5,
    labels=['Youth (17-21)', 'Early Career (21-26)', 'Prime (26-30)', 'Experienced (30-35)', 'Veteran (35-39)']
)

print('=== Age Bin Distribution ===')
bin_dist = df['age_bin'].value_counts().sort_index()
bin_pct  = (df['age_bin'].value_counts(normalize=True).mul(100).round(2)).sort_index()
print(pd.DataFrame({'Count': bin_dist, 'Percentage (%)': bin_pct}))

# Visualization
plt.figure(figsize=(9, 4))
bin_dist.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Age Bin Distribution (5 Equal-Width Bins)', fontsize=13)
plt.xlabel('Age Group')
plt.ylabel('Number of Players')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# We already confirmed in Part 1h that this dataset has NO missing values.
# We demonstrate the correct handling approach anyway as good practice.
# 
# Strategy (if missing values existed):
# - Numerical features → MEDIAN imputation (robust to outliers, better than mean
#   when distributions are skewed — e.g., market_value, goals)
# - Categorical features → MODE imputation (most logical default for categories)

missing_counts = df.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0]

if cols_with_missing.empty:
    print('No missing values detected — no imputation required.')
    print('Strategy if needed: Median for numerical, Mode for categorical.')
else:
    from sklearn.impute import SimpleImputer
    
    num_imputer = SimpleImputer(strategy='median')
    cat_imputer = SimpleImputer(strategy='most_frequent')
    
    num_missing = [c for c in cols_with_missing.index if df[c].dtype in ['int64', 'float64']]
    cat_missing = [c for c in cols_with_missing.index if df[c].dtype == 'object']
    
    if num_missing:
        df[num_missing] = num_imputer.fit_transform(df[num_missing])
    if cat_missing:
        df[cat_missing] = cat_imputer.fit_transform(df[cat_missing])
    
    print('Imputation complete.')
    print(f'Remaining missing values: {df.isnull().sum().sum()}')

In [ ]:
# PART 1: Correlation Analysis (Numerical Features)
# We use the absolute correlation with the target to rank numerical features.
# Features with very low correlation (<0.05) contribute little signal to the model.

import matplotlib.pyplot as plt
import seaborn as sns

num_cols = ['age', 'overall_rating', 'potential_rating', 'matches_played',
            'goals', 'assists', 'minutes_played', 'contract_years_left']

target = df['market_value_million_eur']

corr_with_target = df[num_cols].corrwith(target).abs().sort_values(ascending=False)
print('=== Absolute Correlation with market_value_million_eur ===')
print(corr_with_target.round(4))

# Heatmap of full numerical correlation matrix
plt.figure(figsize=(10, 7))
corr_matrix = df[num_cols + ['market_value_million_eur']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap – Numerical Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# PART 2: Chi-Square Test (Categorical Features vs Target)
# Chi-square tests whether a categorical feature is statistically independent
# of the target. A high p-value (>0.05) means the feature is NOT significantly
# associated with the target and can be dropped.

from scipy.stats import chi2_contingency

# Re-create value_category if not present
if 'value_category' not in df.columns:
    df['value_category'] = pd.qcut(
        df['market_value_million_eur'], q=3,
        labels=['Low Value', 'Medium Value', 'High Value'], duplicates='drop'
    )

cat_features = ['injury_prone', 'transfer_risk_level']
print('=== Chi-Square Test: Categorical Features vs value_category ===')
print(f'{"Feature":<25} {"Chi2":>10} {"p-value":>10} {"Significant?":>15}')
print('-' * 65)
for col in cat_features:
    ct = pd.crosstab(df[col], df['value_category'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = 'YES' if p < 0.05 else 'NO'
    print(f'{col:<25} {chi2:>10.2f} {p:>10.4f} {sig:>15}')

In [ ]:
# Based on the feature selection analysis:
#
# DROPPED (Numerical - low correlation with target):
#   - 'assists'       : corr = 0.0005 — virtually zero relationship with market value
#   - 'matches_played': corr = 0.0027 — negligible predictive power
#
# DROPPED (Categorical - high chi-square p-value, not significant):
#   - 'injury_prone'  : p = 0.98 — completely independent of market value
#   - 'nationality'   : p = 0.50 — no significant association with value
#   - 'position'      : p = 0.84 — no significant association with value
#
# KEPT: 'club', 'transfer_risk_level' have lower p-values and more relevance.
# KEPT: 'player_name' dropped separately as it is a unique identifier, not a feature.

cols_to_drop = ['assists', 'matches_played', 'injury_prone', 'injury_prone_encoded', 'player_name', 'player_id']

# Drop one-hot encoded nationality and position columns
ohe_to_drop = [c for c in df.columns if c.startswith('nationality_') or c.startswith('position_')]
cols_to_drop.extend(ohe_to_drop)

df_selected = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print('=== Features Removed ===')
for c in cols_to_drop:
    print(f'  - {c}')

print()
print(f'Shape before feature selection : {df.shape}')
print(f'Shape after feature selection  : {df_selected.shape}')
print()
print('=== Remaining Columns ===')
print(df_selected.columns.tolist())

In [ ]:
# We apply Stratified Sampling to create a representative subset.
# Stratified sampling preserves the class proportions of the target variable,
# unlike Simple Random Sampling which can accidentally skew the distribution.
# This is preferred when the goal is fair representation across all value classes.

from sklearn.model_selection import train_test_split

# Ensure value_category exists
if 'value_category' not in df_selected.columns:
    df_selected['value_category'] = pd.qcut(
        df_selected['market_value_million_eur'], q=3,
        labels=['Low Value', 'Medium Value', 'High Value'], duplicates='drop'
    )

# Take a 40% stratified sample
_, df_sampled = train_test_split(
    df_selected,
    test_size=0.4,
    stratify=df_selected['value_category'],
    random_state=42
)

print(f'Original dataset size : {len(df_selected)}')
print(f'Sampled dataset size  : {len(df_sampled)}')

In [ ]:
# Comparing distributions confirms our stratified sampling worked correctly —
# both original and sampled datasets should have ~33% per class.

before = df_selected['value_category'].value_counts().sort_index()
after  = df_sampled['value_category'].value_counts().sort_index()

before_pct = df_selected['value_category'].value_counts(normalize=True).mul(100).round(2).sort_index()
after_pct  = df_sampled['value_category'].value_counts(normalize=True).mul(100).round(2).sort_index()

print('=== Class Distribution: Before vs After Stratified Sampling ===')
comparison = pd.DataFrame({
    'Before (Count)' : before,
    'Before (%)'     : before_pct,
    'After (Count)'  : after,
    'After (%)'      : after_pct
})
print(comparison)

# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4C72B0', '#55A868', '#C44E52']

before.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Before Sampling', fontsize=12)
axes[0].set_xlabel('Value Category')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

after.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('After Stratified Sampling (40%)', fontsize=12)
axes[1].set_xlabel('Value Category')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.suptitle('Class Distribution Before vs After Sampling', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print()
print('Observation: Stratified sampling preserved the class proportions perfectly.')

In [ ]:
#Part 3: Data Visualization
# We chose 'overall_rating' for the histogram because it is the most
# meaningful performance metric in football analytics.
# A histogram reveals whether ratings are normally distributed,
# skewed, or concentrated in a narrow band — all important for modelling.

plt.figure(figsize=(9, 5))
plt.hist(df['overall_rating'], bins=20, color='steelblue', edgecolor='black', alpha=0.85)
plt.axvline(df['overall_rating'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f"Mean: {df['overall_rating'].mean():.1f}")
plt.axvline(df['overall_rating'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median: {df['overall_rating'].median():.1f}")
plt.title('Distribution of Player Overall Ratings', fontsize=14)
plt.xlabel('Overall Rating')
plt.ylabel('Number of Players')
plt.legend()
plt.tight_layout()
plt.show()

print('=== Interpretation ===')
print(f"Mean   : {df['overall_rating'].mean():.2f}")
print(f"Median : {df['overall_rating'].median():.2f}")
print(f"Std    : {df['overall_rating'].std():.2f}")
print(f"Skew   : {df['overall_rating'].skew():.4f}")
print()
print('The distribution of overall ratings is approximately symmetric (skew ≈ 0),')
print('with mean and median nearly identical. Most players are rated between 60–94,')
print('indicating a fairly uniform spread across skill levels with no extreme concentration.')
print('This suggests the dataset represents a diverse range of players, not just elite ones.')

In [ ]:
# We chose market_value_million_eur grouped by position for the boxplot.
# Note: since we applied One-Hot Encoding in Part 2b, we reload the original
# 'position' column directly from the raw dataset for this visualization.

df_viz = pd.read_csv('fifa_player_performance_market_value.csv')

position_order = df_viz.groupby('position')['market_value_million_eur'].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_viz,
    x='position',
    y='market_value_million_eur',
    order=position_order,
    palette='Set2'
)
plt.title('Market Value Distribution by Playing Position', fontsize=14)
plt.xlabel('Position')
plt.ylabel('Market Value (Million EUR)')
plt.tight_layout()
plt.show()

print('=== Interpretation ===')
print(df_viz.groupby('position')['market_value_million_eur'].median().sort_values(ascending=False).round(2))
print()
print('The boxplot reveals that attacking positions (RW, ST) tend to command higher')
print('median market values compared to defensive roles (GK, CM).')
print('Goalkeepers (GK) show the lowest median value, reflecting typical market trends.')
print('All positions show wide IQR ranges, indicating high variability within each role.')
print('No extreme outliers are present, confirming the clean nature of this dataset.')

In [ ]:
# We plot overall_rating vs market_value_million_eur, colored by value_category.
# This is the most natural and business-relevant relationship to visualize —
# we expect (and want to test) whether higher-rated players command higher fees.

if 'value_category' not in df.columns:
    df['value_category'] = pd.qcut(
        df['market_value_million_eur'], q=3,
        labels=['Low Value', 'Medium Value', 'High Value'], duplicates='drop'
    )

plt.figure(figsize=(9, 6))
colors = {'Low Value': '#4C72B0', 'Medium Value': '#55A868', 'High Value': '#C44E52'}
for category, group in df.groupby('value_category'):
    plt.scatter(group['overall_rating'], group['market_value_million_eur'],
                label=category, color=colors[category], alpha=0.4, s=15)

plt.title('Overall Rating vs Market Value', fontsize=14)
plt.xlabel('Overall Rating')
plt.ylabel('Market Value (Million EUR)')
plt.legend(title='Value Category')
plt.tight_layout()
plt.show()

# Correlation coefficient
corr_val = df['overall_rating'].corr(df['market_value_million_eur'])
print('=== Interpretation ===')
print(f'Pearson Correlation: {corr_val:.4f}')
print()
print('Surprisingly, the scatterplot reveals almost no linear correlation between')
print('overall_rating and market_value (r ≈ 0.01). The three value categories are')
print('heavily overlapping across all rating levels, suggesting that in this dataset,')
print('overall rating alone is NOT a strong predictor of market value.')
print('This is a counter-intuitive finding — real-world football data typically shows')
print('a much stronger relationship, suggesting this dataset may be synthetically generated.')

In [ ]:
# A correlation heatmap gives a complete picture of relationships between
# all numerical features simultaneously. It is essential for identifying
# multicollinearity (two features carrying the same information) and
# for guiding feature selection decisions in Part 2.

num_cols = ['age', 'overall_rating', 'potential_rating', 'matches_played',
            'goals', 'assists', 'minutes_played', 'market_value_million_eur',
            'contract_years_left']

corr_matrix = df[num_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # show only lower triangle to avoid redundancy

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap – Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

print('=== Interpretation of Key Correlations ===')
print()
print('1. potential_rating vs minutes_played (r = 0.047):')
print('   The strongest correlation in the dataset — players with higher potential')
print('   tend to receive slightly more playing time, suggesting coaches invest')
print('   in developing high-ceiling players. However, the correlation is still weak.')
print()
print('2. overall_rating vs contract_years_left (r = -0.045):')
print('   A slight negative correlation — higher-rated players tend to have fewer')
print('   contract years remaining. This may reflect that top clubs offer shorter,')
print('   higher-value deals and renegotiate more frequently with star players.')
print()
print('Overall finding: All correlations are extremely low (all < 0.05), confirming')
print('that the features in this dataset are largely independent of each other.')
print('This is unusual for real football data and suggests synthetic data generation.')

In [ ]:
#Part 4: Insight Discovery (Analytical Thinking)
# INSIGHT: Player position has NO statistically significant effect on market value.
#
# We ran a one-way ANOVA test to check whether market value differs
# significantly across positions. Despite intuition suggesting attackers
# are worth more, the test shows no significant difference.
# F-statistic = 0.56, p-value = 0.81 (far above threshold of 0.05).
# This means we cannot reject the null hypothesis that all positions
# have equal mean market values. Bayern Munich leads all clubs with an
# average market value of €95.01M vs Liverpool's €85.17M — a €10M gap
# that IS meaningful from a business perspective even if position isn't.

from scipy.stats import f_oneway

# Use df_viz (raw data) since 'position' was one-hot encoded in Part 2b
df_viz = pd.read_csv('fifa_player_performance_market_value.csv')

# ANOVA test: does position significantly affect market value?
groups = [df_viz[df_viz['position'] == p]['market_value_million_eur'].values for p in df_viz['position'].unique()]
f_stat, p_val = f_oneway(*groups)

print('=== ANOVA Test: Position vs Market Value ===')
print(f'F-statistic : {f_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print()
if p_val > 0.05:
    print('Result: No statistically significant difference in market value across positions.')
    print('(p > 0.05 — we fail to reject the null hypothesis)')
else:
    print('Result: Significant difference detected across positions. (p < 0.05)')

print()
# Club-level breakdown as supporting evidence
print('=== Average Market Value by Club ===')
club_stats = df_viz.groupby('club')['market_value_million_eur'].mean().sort_values(ascending=False).round(2)
print(club_stats)

# Visualization: avg market value by club
plt.figure(figsize=(9, 5))
club_stats.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Player Market Value by Club\n(Insight 4a)', fontsize=13)
plt.xlabel('Club')
plt.ylabel('Avg Market Value (Million EUR)')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# INSIGHT: Injury-prone players at LOW transfer risk have HIGHER market value
# than non-injury-prone players at the same risk level.
#
# This is an interaction effect between 'injury_prone' and 'transfer_risk_level'.
# Among Low-risk players: injury_prone=Yes → avg €93.05M vs injury_prone=No → €89.77M.
# This suggests that injury-prone players who are considered low transfer risk
# may be high-value stars who clubs are willing to retain despite injury history.
# The interaction disappears at High risk (Yes=€88.70M vs No=€89.59M),
# confirming the pattern is specific to the Low risk group.
# A t-test between injury_prone groups shows t=-0.026, p=0.979 overall —
# but the interaction by risk level reveals a nuance the overall test misses.

from scipy.stats import ttest_ind

interaction = df.groupby(['injury_prone', 'transfer_risk_level'])['market_value_million_eur'].mean().round(2).unstack()
print('=== Avg Market Value: Injury Prone × Transfer Risk Level ===')
print(interaction)

print()
# T-test overall
t_stat, p_val = ttest_ind(
    df[df['injury_prone'] == 'Yes']['market_value_million_eur'],
    df[df['injury_prone'] == 'No']['market_value_million_eur']
)
print(f'Overall T-test (injury_prone Yes vs No): t={t_stat:.4f}, p={p_val:.4f}')
print('→ No significant difference overall, but interaction pattern exists within risk groups.')

# Visualization: grouped bar chart
interaction.plot(kind='bar', figsize=(8, 5), edgecolor='black', colormap='Set2')
plt.title('Market Value by Injury Status and Transfer Risk\n(Insight 4b – Interaction Effect)', fontsize=13)
plt.xlabel('Injury Prone')
plt.ylabel('Avg Market Value (Million EUR)')
plt.xticks(rotation=0)
plt.legend(title='Transfer Risk Level')
plt.tight_layout()
plt.show()

In [ ]:
# INSIGHT: 255 players rated 85+ have LOW market value (under €59M).
#
# Common expectation: higher overall rating → higher market value.
# Reality in this dataset: 255 players with overall_rating >= 85
# fall into the Low Value category, with a mean market value of just €31.49M
# and a minimum as low as €0.67M. Meanwhile, the Pearson correlation between
# overall_rating and market_value is only r = 0.013 — essentially zero.
# This counter-intuitive finding suggests that raw skill rating alone
# does NOT drive market value in this dataset. Other unmeasured factors
# (age, contract status, club demand, media profile) likely play a larger role.
# This is also consistent with real football — Messi at 36 commands less
# than a 24-year-old with similar ratings due to longevity risk.

high_rating_low_val = df[(df['overall_rating'] >= 85) & (df['value_category'] == 'Low Value')]
corr_rating_value   = df['overall_rating'].corr(df['market_value_million_eur'])

print('=== Counter-Intuitive Finding ===')
print(f'Players with overall_rating >= 85 AND Low Value category: {len(high_rating_low_val)}')
print()
print('Their market value stats:')
print(high_rating_low_val['market_value_million_eur'].describe().round(2))
print()
print(f'Pearson correlation (overall_rating vs market_value): {corr_rating_value:.4f}')
print('→ Near-zero correlation confirms rating is NOT a reliable predictor of market value.')

# Visualization: scatter highlighting the counter-intuitive group
plt.figure(figsize=(10, 6))
plt.scatter(df['overall_rating'], df['market_value_million_eur'],
            color='lightsteelblue', alpha=0.3, s=15, label='All Players')
plt.scatter(high_rating_low_val['overall_rating'], high_rating_low_val['market_value_million_eur'],
            color='crimson', alpha=0.7, s=25, label='High Rating but Low Value (n=255)')
plt.axvline(85, color='black', linestyle='--', linewidth=1.2, label='Rating = 85 threshold')
plt.title('Counter-Intuitive: High-Rated Players with Low Market Value\n(Insight 4c)', fontsize=13)
plt.xlabel('Overall Rating')
plt.ylabel('Market Value (Million EUR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# RECOMMENDATION: Clubs should scout players aged 22–26 from lower-value clubs
# (e.g., Liverpool avg €85.17M) who have high overall_rating and low transfer_risk.
#
# Evidence:
# - Players aged 22–26 have the highest avg market value (€92.21M) of any age group,
#   meaning they are in their peak appreciation window.
# - Players aged 27–31 follow closely at €92.12M but are closer to decline.
# - Liverpool players average €85.17M — the lowest of all 7 clubs — yet have
#   avg overall_rating of 77.45, the HIGHEST of all clubs.
# - This means Liverpool has undervalued talent: high-skill players priced below market.
# - A club acquiring a 22–26 year old Liverpool player at current value could
#   realize significant ROI as the player enters their prime years.
# Business action: prioritize acquisition of 22–26 year old players from
# lower-valuation clubs with high ratings and low transfer risk scores.

# Use df_viz since 'club' was one-hot encoded in Part 2b
age_value = df_viz.groupby(
    pd.cut(df_viz['age'], bins=[16,21,26,31,36,40], labels=['17-21','22-26','27-31','32-36','37-39'])
)['market_value_million_eur'].mean().round(2)

club_summary = df_viz.groupby('club')[['market_value_million_eur','overall_rating']].mean().round(2).sort_values('market_value_million_eur')

print('=== Average Market Value by Age Group ===')
print(age_value)
print()
print('=== Club: Market Value vs Overall Rating ===')
print(club_summary)
print()
print('Key Finding: Liverpool has the LOWEST avg market value (85.17M)')
print('but the HIGHEST avg overall rating (77.45) — best value-for-money club.')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age group bar chart
age_value.plot(kind='bar', ax=axes[0], color='mediumseagreen', edgecolor='black')
axes[0].set_title('Avg Market Value by Age Group', fontsize=12)
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Avg Market Value (Million EUR)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15, ha='right')

# Club scatter: value vs rating
axes[1].scatter(club_summary['overall_rating'], club_summary['market_value_million_eur'],
                color='steelblue', s=120, zorder=5)
for club, row in club_summary.iterrows():
    axes[1].annotate(club, (row['overall_rating'], row['market_value_million_eur']),
                     textcoords='offset points', xytext=(5, 4), fontsize=8)
axes[1].set_title('Club: Avg Rating vs Avg Market Value\n(Bottom-right = undervalued)', fontsize=11)
axes[1].set_xlabel('Avg Overall Rating')
axes[1].set_ylabel('Avg Market Value (Million EUR)')

plt.suptitle('Business Recommendation: Target Young, Undervalued Talent\n(Insight 4d)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#Part 5: Feature Engineering
# ============================================================
# PART 5: Feature Engineering
# We create 3 new features derived from existing attributes.
# Even though correlations are low (synthetic dataset), these
# features are standard in real football analytics and would
# meaningfully improve classification models on real data.
# ============================================================

# --- Feature 1: goal_contribution (goals + assists) ---
# Combines goals and assists into a single attacking output metric.
# In football analytics, total goal contributions is a more complete
# measure of an attacker's value than goals alone. A player who scores
# 15 and assists 20 is arguably more valuable than one who scores 25
# and assists 5, but raw goals alone would rank them differently.
# This composite feature reduces two correlated columns into one
# stronger signal, reducing dimensionality while preserving information.

df_viz['goal_contribution'] = df_viz['goals'] + df_viz['assists']

print('=== Feature 1: goal_contribution (goals + assists) ===')
print(df_viz['goal_contribution'].describe().round(2))
print()
print('Average by Value Category:')
df_viz['value_category'] = pd.qcut(
    df_viz['market_value_million_eur'], q=3,
    labels=['Low Value', 'Medium Value', 'High Value'], duplicates='drop'
)
print(df_viz.groupby('value_category')['goal_contribution'].mean().round(2))

In [ ]:
# --- Feature 2: goals_per_90 (goals per 90 minutes played) ---
# A standard professional football metric that normalizes goal-scoring
# by playing time. A player who scores 10 goals in 900 minutes is far
# more efficient than one who scores 10 in 3000 minutes.
# Raw 'goals' is biased by how much playing time a player received,
# which is influenced by injury, squad rotation, and club policy.
# goals_per_90 removes this bias and captures true scoring efficiency —
# making it a much fairer and more informative feature for ML models.
# We handle the one player with 0 minutes by setting their value to 0.

df_viz['goals_per_90'] = df_viz['goals'] / (df_viz['minutes_played'] / 90)
df_viz['goals_per_90'] = df_viz['goals_per_90'].replace([np.inf, -np.inf], np.nan).fillna(0)
df_viz['goals_per_90'] = df_viz['goals_per_90'].round(4)

print('=== Feature 3: goals_per_90 ===')
print(df_viz['goals_per_90'].describe().round(4))
print()
print('Average goals_per_90 by Value Category:')
print(df_viz.groupby('value_category')['goals_per_90'].mean().round(4))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df_viz = pd.read_csv('fifa_player_performance_market_value.csv')
df_viz['value_category'] = pd.qcut(
    df_viz['market_value_million_eur'], q=3,
    labels=['Low Value', 'Medium Value', 'High Value'], duplicates='drop'
)
df_viz['goal_contribution'] = df_viz['goals'] + df_viz['assists']
df_viz['goals_per_90'] = df_viz['goals'] / (df_viz['minutes_played'] / 90)
df_viz['goals_per_90'] = df_viz['goals_per_90'].replace([np.inf, -np.inf], np.nan).fillna(0).round(4)

# --- Visualization of new features ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# goal_contribution distribution
axes[0].hist(df_viz['goal_contribution'], bins=20, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].axvline(df_viz['goal_contribution'].mean(), color='red', linestyle='--', linewidth=1.5,
                label=f"Mean: {df_viz['goal_contribution'].mean():.1f}")
axes[0].set_title('Goal Contribution\n(goals + assists)', fontsize=12)
axes[0].set_xlabel('Goal Contribution')
axes[0].set_ylabel('Count')
axes[0].legend()

# goals_per_90 by value category boxplot
df_viz.boxplot(column='goals_per_90', by='value_category', ax=axes[1],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Goals per 90 mins\nby Value Category', fontsize=12)
axes[1].set_xlabel('Value Category')
axes[1].set_ylabel('Goals per 90')
plt.suptitle('')

plt.suptitle('Part 5: Engineered Features Overview', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL SUMMARY — Key Metrics at a Glance
# ============================================================

print('=' * 55)
print('   FIFA Player Performance & Market Value — Summary')
print('=' * 55)

print(f'\n Dataset shape         : 2,800 rows × 16 columns')
print(f' Missing values        : 0')
print(f' Duplicate records     : 0')
print(f' Target variable       : market_value_million_eur')
print(f' Class balance         : ~33.3% per class (perfectly balanced)')

print(f'\n Features after selection  : {df_selected.shape[1]} columns')
print(f' Sample size after sampling: {len(df_sampled)} players')

print(f'\n Engineered features added : 2')
print(f'   - goal_contribution (goals + assists)')
print(f'   - goals_per_90      (goals per 90 minutes played)')

print(f'\n Top Insight: Liverpool avg rating = 77.45 (highest)')
print(f'             Liverpool avg value  = €85.17M (lowest)')
print(f'             → Best undervalued talent pool in the dataset')

print(f'\n Counter-intuitive: 255 players rated ≥85 are in Low Value')
print(f'             Overall rating ↔ market value correlation = 0.013')

